# **Latin Morphological Analyser** - Dataset Preparation

This notebook parses the PROIEL XML, builds label vocabularies, splits the data into train/val/test, and tokenises everything. The processed datasets and vocabularies are saved to Drive so the other notebooks (training, evaluation, confidence threshold) can load them.

Dataset used:  
https://github.com/syntacticus/syntacticus-treebank-data  
https://dev.syntacticus.org/development-guide/#lemma-part-of-speech-and-morphology


## **Set Up Environment**

In [1]:
import os

### Install Latin BERT

Download the pre-trained Latin BERT model from the GitHub repository and define the path for the model to be used for fine-tuning.

In [2]:
# clone latin bert repo
!git clone https://github.com/dbamman/latin-bert.git
%cd latin-bert

Cloning into 'latin-bert'...
remote: Enumerating objects: 154, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 154 (delta 4), reused 27 (delta 3), pack-reused 122 (from 1)
Receiving objects: 100% (154/154), 6.77 MiB | 14.26 MiB/s, done.
Resolving deltas: 100% (59/59), done.
/content/latin-bert


In [3]:
# Download pre-trained BERT model for Latin
!./scripts/download.sh

--2026-05-16 15:25:24--  https://drive.usercontent.google.com/download?export=download&id=1Te_14UB-DZ8wYPhHGyDg7LadDTjNzpti&confirm=t&uuid=da2956f9-3ecd-4201-afee-049857f3f1a9
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 74.125.69.132, 2607:f8b0:4001:c08::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|74.125.69.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 448020480 (427M) [application/octet-stream]
Saving to: ‘latin_bert.tar’

latin_bert.tar      100%[===================>] 427.27M  39.8MB/s    in 6.9s    

2026-05-16 15:25:32 (61.5 MB/s) - ‘latin_bert.tar’ saved [448020480/448020480]



In [4]:
!ls -lah models/latin_bert | head

total 428M
drwxrwxr-x 2 1001 1001 4.0K Sep  4  2020 .
drwxr-xr-x 4 root root 4.0K May 16 15:25 ..
-rw-rw-r-- 1 1001 1001  503 Sep  4  2020 config.json
-rw-rw-r-- 1 1001 1001 428M Sep  4  2020 pytorch_model.bin
-rw-rw-r-- 1 1001 1001 217K Sep  4  2020 vocab.txt


In [5]:
%cd ..

/content


In [6]:
MODEL_PATH = os.path.join("latin-bert", "models", "latin_bert")

### Import Dependencies

In [7]:
import sys
from dataclasses import dataclass, field
from typing import Optional
import xml.etree.ElementTree as ET
from collections import defaultdict
import transformers
from transformers import AutoTokenizer
import datasets
from datasets import Dataset
import json

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
# load constants defined in morph_constants.py
sys.path.append('/content/drive/MyDrive/FYP')
from morph_constants import MORPH_POSITIONS, FEATURE_ORDER, ALL_FEATS, IGNORE_INDEX

In [10]:
print("Python:", sys.version.split()[0])
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)

Python: 3.12.13
Transformers: 5.0.0
Datasets: 4.0.0


## **Prepare Data**

### Load Dataset

Parse the XML file into sentences containing tokens.

In [11]:
DATA_PATH = '/content/drive/MyDrive/FYP/latin-nt.xml'

In [12]:
# define token and sentence classes

@dataclass
class Token:
  id: str
  form: Optional[str]
  lemma: Optional[str]
  pos: Optional[str]
  morphology: Optional[str]
  head_id: Optional[str]
  relation: Optional[str]

@dataclass
class Sentence:
  id: str
  tokens: list[Token] = field(default_factory=list)


In [13]:
# parses all sentences in xml file, returns a list containing the sentences

def parse_proiel_xml(filepath: str) -> list[Sentence]:
  tree = ET.parse(filepath)
  root = tree.getroot()
  sentences = []

  for sentence_el in root.iter("sentence"):
    sent = Sentence(id=sentence_el.get("id"))
    for token_el in sentence_el.findall("token"):
      # Skip empty/trace tokens (no 'form')
      if token_el.get("form") is None:
        continue
      sent.tokens.append(Token(
          id=token_el.get("id"),
          form=token_el.get("form"),
          lemma=token_el.get("lemma"),
          pos=token_el.get("part-of-speech"),
          morphology=token_el.get("morphology"),
          head_id=token_el.get("head-id"),
          relation=token_el.get("relation"),
      ))
    if sent.tokens:
      sentences.append(sent)

  return sentences


In [14]:
# load all sentences
sentences = parse_proiel_xml(DATA_PATH)
print(f"Loaded {len(sentences)} sentences")

total_words = sum(len(s.tokens) for s in sentences)
print(f"Total word tokens: {total_words:,}")

print(f"Avg sentence length: {total_words / len(sentences):.1f}")

Loaded 11851 sentences
Total word tokens: 110,774
Avg sentence length: 9.3


In [15]:
sentences[0]

Sentence(id='12667', tokens=[Token(id='250021', form='liber', lemma='liber', pos='Nb', morphology='-s---mn--i', head_id='851355', relation='xobj'), Token(id='250022', form='generationis', lemma='generatio', pos='Nb', morphology='-s---fg--i', head_id='250021', relation='atr'), Token(id='250023', form='Iesu', lemma='Iesus', pos='Ne', morphology='-s---mg--i', head_id='250022', relation='atr'), Token(id='250024', form='Christi', lemma='Christus', pos='Ne', morphology='-s---mg--i', head_id='250023', relation='apos'), Token(id='250025', form='filii', lemma='filius', pos='Nb', morphology='-s---mg--i', head_id='250023', relation='apos'), Token(id='250026', form='David', lemma='David', pos='Ne', morphology='---------n', head_id='250025', relation='atr'), Token(id='250027', form='filii', lemma='filius', pos='Nb', morphology='-s---mg--i', head_id='250025', relation='apos'), Token(id='250028', form='Abraham', lemma='Abraham', pos='Ne', morphology='---------n', head_id='250027', relation='atr')])

### Extract Labels from Data

**PROIEL morphology string decoder**  

The PROIEL `morphology` field is a 10-char positional string.  
Each character encodes one feature. Unused positions are "-".  
Reference: https://dev.syntacticus.org/development-guide/#lemma-part-of-speech-and-morphology

<br>

The final 2 characters ("strength" and "inflection") will be ignored.
*   no records contain "strength" info
*   "non-inflecting" tokens contain no other morphological information



In [16]:
def decode_morphology(morph_str: str) -> dict:
    """
    Decode a PROIEL morphology string into a named feature dict.
    e.g. "3s---mn-" -> {"person":"3", "number":"Sing", ..., "case":"Nom", ...}
    Returns "None" for any absent/inapplicable position.
    """
    result = {feat: "None" for _, (feat, _) in MORPH_POSITIONS.items()}
    if not morph_str:
        return result
    for pos, (feat, mapping) in MORPH_POSITIONS.items():
        if pos < len(morph_str):
            result[feat] = mapping.get(morph_str[pos], "None")
    return result

**Build label vocabularies from data**

In [17]:
def build_label_vocabs(sentences):
  """
  Scan all sentences and collect every observed value per feature.
  """
  vocabs = defaultdict(set)
  vocabs["pos"] = set()

  for sent in sentences:
    for tok in sent.tokens:
      if not tok.pos:
        continue
      vocabs["pos"].add(tok.pos)
      feats = decode_morphology(tok.morphology)
      for feat, val in feats.items():
        vocabs[feat].add(val)

  # Build label2id / id2label dicts; keep "None" last in each vocab
  label2id_all, id2label_all = {}, {}
  for feat, vals in vocabs.items():
    sorted_vals = sorted(v for v in vals if v != "None")
    if "None" in vals:
      sorted_vals.append("None")
    label2id_all[feat] = {v: i for i, v in enumerate(sorted_vals)}
    id2label_all[feat] = {i: v for v, i in label2id_all[feat].items()}

  return label2id_all, id2label_all

In [18]:
# Build label vocabularies from data.
label2id_all, id2label_all = build_label_vocabs(sentences)

# Quick summary
for feat in ALL_FEATS:
  print(f"{feat:10s} ({len(label2id_all[feat])} labels): {list(label2id_all[feat].keys())}")

pos        (23 labels): ['A-', 'C-', 'Df', 'Dq', 'Du', 'F-', 'G-', 'I-', 'Ma', 'Mo', 'Nb', 'Ne', 'Pc', 'Pd', 'Pi', 'Pk', 'Pp', 'Pr', 'Ps', 'Pt', 'Px', 'R-', 'V-']
person     (4 labels): ['1', '2', '3', 'None']
number     (3 labels): ['Plur', 'Sing', 'None']
tense      (7 labels): ['Fut', 'FutPerf', 'Imp', 'Perf', 'Plup', 'Pres', 'None']
mood       (9 labels): ['Gdv', 'Ger', 'Imp', 'Ind', 'Inf', 'Part', 'Sub', 'Sup', 'None']
voice      (3 labels): ['Act', 'Pass', 'None']
gender     (8 labels): ['Fem', 'FemNeut', 'Masc', 'MascFem', 'MascFemNeut', 'MascNeut', 'Neut', 'None']
case       (7 labels): ['Abl', 'Acc', 'Dat', 'Gen', 'Nom', 'Voc', 'None']
degree     (4 labels): ['Comp', 'Pos', 'Sup', 'None']


**Convert sentences to multi-label records**

In [19]:
def to_multilabel_records(sentences, label2id_all):
    records = []
    for sent in sentences:
        tokens, pos_labels = [], []
        feat_labels = {feat: [] for feat in FEATURE_ORDER}
        valid = True

        for tok in sent.tokens:
            if not tok.pos or tok.pos not in label2id_all["pos"]:
                valid = False
                break
            tokens.append(tok.form.lower())
            pos_labels.append(label2id_all["pos"][tok.pos])

            feats = decode_morphology(tok.morphology)
            for feat in FEATURE_ORDER:
                feat_labels[feat].append(label2id_all[feat][feats[feat]])

        if valid and len(tokens) == len(pos_labels):
            record = {"tokens": tokens, "labels_pos": pos_labels}
            for feat in FEATURE_ORDER:
                record[f"labels_{feat}"] = feat_labels[feat]
            records.append(record)

    return records

In [20]:
multilabel_records = to_multilabel_records(sentences, label2id_all)
print(f"Total records: {len(multilabel_records)}")

Total records: 11851


### Split into Train/Test/Validation Sets

In [21]:
# 70% train, 15% test, 15% val
multi_dataset = Dataset.from_list(multilabel_records)
multi_dataset = multi_dataset.train_test_split(test_size=0.3, seed=42)
train_ds = multi_dataset["train"]
test_val = multi_dataset["test"].train_test_split(test_size=0.5, seed=42)
test_ds  = test_val["train"]
val_ds   = test_val["test"]

In [22]:
train_ds[0]

{'tokens': ['domine', 'bonum', 'est', 'nos', 'hic', 'esse'],
 'labels_pos': [10, 0, 22, 16, 2, 22],
 'labels_person': [3, 3, 2, 0, 3, 3],
 'labels_number': [1, 1, 1, 0, 2, 2],
 'labels_tense': [6, 6, 5, 6, 6, 5],
 'labels_mood': [8, 8, 3, 8, 8, 4],
 'labels_voice': [2, 2, 0, 2, 2, 0],
 'labels_gender': [2, 6, 7, 2, 7, 7],
 'labels_case': [5, 4, 6, 1, 6, 6],
 'labels_degree': [3, 1, 3, 3, 3, 3]}

In [23]:
print(f"{len(train_ds)} train | {len(val_ds)} val | {len(test_ds)} test")

8295 train | 1778 val | 1778 test


In [24]:
for name, ds in [("train", train_ds), ("val", val_ds), ("test", test_ds)]:
    n = sum(len(ex["tokens"]) for ex in ds)
    print(f"{name}: {len(ds):,} sentences | {n:,} word tokens")

train: 8,295 sentences | 77,534 word tokens
val: 1,778 sentences | 16,706 word tokens
test: 1,778 sentences | 16,534 word tokens


## **Tokenization**

### Load Tokenizer

In [25]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

In [26]:
type(tokenizer)

transformers.models.bert.tokenization_bert.BertTokenizer

In [27]:
# tokenize function
def tokenize_and_align_multilabels(batch):
    tokenized = tokenizer(
        batch["tokens"],
        truncation=True,
        is_split_into_words=True,
        padding="max_length",
        max_length=128,
    )

    all_aligned = {feat: [] for feat in ALL_FEATS}

    for i in range(len(batch["tokens"])):
        word_ids = tokenized.word_ids(batch_index=i)
        prev_wid = None
        per_feat = {feat: [] for feat in ALL_FEATS}

        for wid in word_ids:
            is_first = (wid is not None and wid != prev_wid)
            for feat in ALL_FEATS:
                word_labels = batch[f"labels_{feat}"][i]
                if wid is None:
                    per_feat[feat].append(IGNORE_INDEX)
                elif is_first:
                    per_feat[feat].append(word_labels[wid])
                else:
                    per_feat[feat].append(IGNORE_INDEX)
            prev_wid = wid

        for feat in ALL_FEATS:
            all_aligned[feat].append(per_feat[feat])

    for feat in ALL_FEATS:
        tokenized[f"labels_{feat}"] = all_aligned[feat]

    return tokenized

### Tokenize Dataset

In [28]:
# Columns to drop — "tokens" (strings) and original word-level label lists
# The aligned labels_{feat} overwrite the word-level ones with the same key,
# but "tokens" has no counterpart to overwrite it, so it must be removed.
cols_to_remove = ["tokens"] + [f"labels_{feat}" for feat in ALL_FEATS]

In [29]:
train_ds = train_ds.map(tokenize_and_align_multilabels, batched=True, remove_columns=cols_to_remove)
val_ds = val_ds.map(tokenize_and_align_multilabels,   batched=True, remove_columns=cols_to_remove)
test_ds = test_ds.map(tokenize_and_align_multilabels,  batched=True, remove_columns=cols_to_remove)

Map:   0%|          | 0/8295 [00:00<?, ? examples/s]

Map:   0%|          | 0/1778 [00:00<?, ? examples/s]

Map:   0%|          | 0/1778 [00:00<?, ? examples/s]

In [30]:
# Verify only tensor-compatible columns remain
print(train_ds.column_names)
# Expected: ['input_ids', 'attention_mask', 'token_type_ids',
#            'labels_pos', 'labels_person', ..., 'labels_degree']

['labels_pos', 'labels_person', 'labels_number', 'labels_tense', 'labels_mood', 'labels_voice', 'labels_gender', 'labels_case', 'labels_degree', 'input_ids', 'token_type_ids', 'attention_mask']


In [31]:
print(train_ds[0])

{'labels_pos': [-100, 10, 0, 22, 16, 2, 22, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100], 'labels_person': [-100, 3, 3, 2, 0, 3, 3, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -

## **Save Processed Data**

Save the tokenized datasets and label vocabularies to Drive so the training, evaluation, and confidence-threshold notebooks can load them.

In [32]:
# Save tokenized datasets and label vocabularies to Drive so the
# training, evaluation, and confidence-threshold notebooks can load them.

PROCESSED_DIR = "/content/drive/MyDrive/FYP/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

train_ds.save_to_disk(f"{PROCESSED_DIR}/train_ds")
val_ds.save_to_disk(f"{PROCESSED_DIR}/val_ds")
test_ds.save_to_disk(f"{PROCESSED_DIR}/test_ds")

with open(f"{PROCESSED_DIR}/label2id_all.json", "w") as f:
    json.dump(label2id_all, f, indent=2)

# id2label_all has int keys; convert to str for JSON
with open(f"{PROCESSED_DIR}/id2label_all.json", "w") as f:
    json.dump({feat: {str(k): v for k, v in d.items()} for feat, d in id2label_all.items()}, f, indent=2)

print("Saved processed data to", PROCESSED_DIR)

Saving the dataset (0/1 shards):   0%|          | 0/8295 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1778 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1778 [00:00<?, ? examples/s]

Saved processed data to /content/drive/MyDrive/FYP/processed
